# 12. Debugging Techniques Practice

这一练习覆盖张量健康检查、非有限梯度定位和梯度范数统计。

In [ ]:
import math
import torch
import torch.nn as nn


In [ ]:
def tensor_health(tensor):
    finite = torch.isfinite(tensor)
    stats = {
        'shape': tuple(tensor.shape),
        'dtype': str(tensor.dtype).replace('torch.', ''),
        'finite': bool(finite.all().item()) if tensor.numel() else True,
        'num_nonfinite': int((~finite).sum().item()),
    }
    if tensor.numel() and finite.any():
        valid = tensor[finite]
        stats['min'] = float(valid.min().item())
        stats['max'] = float(valid.max().item())
        stats['mean'] = float(valid.mean().item())
    else:
        stats['min'] = float('nan')
        stats['max'] = float('nan')
        stats['mean'] = float('nan')
    return stats


def has_nonfinite(tensor):
    return not bool(torch.isfinite(tensor).all().item())


def gradient_l2_norm(parameters):
    total = 0.0
    for param in parameters:
        if param.grad is None:
            continue
        total += float(param.grad.detach().pow(2).sum().item())
    return math.sqrt(total)


def find_nonfinite_grad_names(named_parameters):
    bad = []
    for name, param in named_parameters:
        if param.grad is not None and not torch.isfinite(param.grad).all():
            bad.append(name)
    return bad


In [ ]:
def test_tensor_health():
    x = torch.tensor([1.0, 2.0, float('nan')])
    stats = tensor_health(x)
    assert stats['shape'] == (3,)
    assert stats['finite'] is False
    assert stats['num_nonfinite'] == 1
    assert has_nonfinite(x) is True
    print('✅ tensor_health 通过')


def test_gradient_l2_norm():
    layer = nn.Linear(2, 1)
    layer.weight.grad = torch.tensor([[3.0, 4.0]])
    layer.bias.grad = torch.tensor([12.0])
    norm = gradient_l2_norm(layer.parameters())
    assert abs(norm - 13.0) < 1e-6
    print('✅ gradient_l2_norm 通过')


def test_find_nonfinite_grad_names():
    layer = nn.Linear(2, 1)
    layer.weight.grad = torch.tensor([[1.0, float('nan')]])
    layer.bias.grad = torch.tensor([0.0])
    bad = find_nonfinite_grad_names(layer.named_parameters())
    assert bad == ['weight']
    print('✅ find_nonfinite_grad_names 通过')


test_tensor_health()
test_gradient_l2_norm()
test_find_nonfinite_grad_names()


In [ ]:
layer = nn.Linear(2, 1)
layer.weight.grad = torch.tensor([[1.0, 2.0]])
layer.bias.grad = torch.tensor([3.0])
print('梯度范数:', gradient_l2_norm(layer.parameters()))
print('张量健康:', tensor_health(torch.tensor([[1.0, 2.0], [3.0, 4.0]])))
